In [ ]:
# =========================
# 📌 CELL 1: Imports + Dataset
# =========================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

# =========================
# Dataset Paths
# =========================
# Assumes the notebook is run from notebooks/ (Jupyter's default cwd).
# Download the dataset per data/README.md before running.
base_dir = os.path.join("..", "data", "split_dataset")

train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")
test_dir  = os.path.join(base_dir, "test")

# =========================
# Load Datasets
# =========================
IMG_SIZE = (224,224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

NUM_CLASSES = len(train_ds.class_names)

print("✅ Classes:", train_ds.class_names)


In [2]:
# =========================
# 📌 CELL 2: Hybrid Model
# =========================


# =========================
# CBAM Block
# =========================
def cbam_block(x, ratio=8):

    channel = x.shape[-1]

    avg_pool = layers.GlobalAveragePooling2D()(x)
    max_pool = layers.GlobalMaxPooling2D()(x)

    shared = layers.Dense(channel//ratio, activation='relu')

    avg = shared(avg_pool)
    max = shared(max_pool)

    channel_att = layers.Dense(channel, activation='sigmoid')(avg+max)
    channel_att = layers.Reshape((1,1,channel))(channel_att)

    x = layers.Multiply()([x,channel_att])


    avg_sp = layers.Lambda(lambda z: tf.reduce_mean(z,axis=-1,keepdims=True))(x)
    max_sp = layers.Lambda(lambda z: tf.reduce_max(z,axis=-1,keepdims=True))(x)

    concat = layers.Concatenate()([avg_sp,max_sp])

    spatial_att = layers.Conv2D(1,7,padding='same',activation='sigmoid')(concat)

    x = layers.Multiply()([x,spatial_att])

    return x


# =========================
# CNN Branch (EfficientNet)
# =========================
def create_cnn_cbam(inputs):

    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    base.trainable = False

    x = base.output

    x = cbam_block(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512,activation='relu')(x)

    return x


# =========================
# Custom ViT Branch
# =========================
def create_vit(inputs,
               patch=16,
               dim=64,
               heads=4,
               layers_n=4):

    patches = layers.Conv2D(
        dim,
        kernel_size=patch,
        strides=patch,
        padding="valid"
    )(inputs)

    n_patches = (224//patch)*(224//patch)

    x = layers.Reshape((n_patches,dim))(patches)


    pos = tf.range(start=0,limit=n_patches,delta=1)

    pos_embed = layers.Embedding(
        input_dim=n_patches,
        output_dim=dim
    )(pos)

    x = x + pos_embed


    for _ in range(layers_n):

        attn = layers.MultiHeadAttention(
            num_heads=heads,
            key_dim=dim
        )(x,x)

        x = layers.Add()([x,attn])
        x = layers.LayerNormalization()(x)

        ffn = layers.Dense(dim*2,activation="relu")(x)
        ffn = layers.Dense(dim)(ffn)

        x = layers.Add()([x,ffn])
        x = layers.LayerNormalization()(x)


    x = layers.Flatten()(x)

    x = layers.Dense(512,activation="relu")(x)

    return x


# =========================
# Hybrid Model
# =========================
def build_hybrid(num_classes):

    inputs = layers.Input(shape=(224,224,3))


    cnn_feat = create_cnn_cbam(inputs)

    vit_feat = create_vit(inputs)


    fusion = layers.Concatenate()([cnn_feat,vit_feat])

    x = layers.Dense(1024,activation="relu")(fusion)

    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes,activation="softmax")(x)


    model = Model(inputs,outputs)

    return model


model = build_hybrid(NUM_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling (Rescaling)         │ (None, 224, 224, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ normalization (Normalization) │ (None, 224, 224, 3)       │               7 │ rescaling[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_1 (Rescaling)       │ (None, 224, 224, 3)       │               0 │ normalization[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv_pad (ZeroPadding2D) │ (None, 225, 225, 3)       │               0 │ rescaling_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_conv (Conv2D)            │ (None, 112, 112, 32)      │             864 │ stem_conv_pad[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_bn (BatchNormalization)  │ (None, 112, 112, 32)      │             128 │ stem_conv[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ stem_activation (Activation)  │ (None, 112, 112, 32)      │               0 │ stem_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_dwconv                │ (None, 112, 112, 32)      │             288 │ stem_activation[0][0]      │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_bn                    │ (None, 112, 112, 32)      │             128 │ block1a_dwconv[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_activation            │ (None, 112, 112, 32)      │               0 │ block1a_bn[0][0]           │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_squeeze            │ (None, 32)                │               0 │ block1a_activation[0][0]   │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reshape (Reshape)  │ (None, 1, 1, 32)          │               0 │ block1a_se_squeeze[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_reduce (Conv2D)    │ (None, 1, 1, 8)           │             264 │ block1a_se_reshape[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1a_se_expand (Conv2D)    │ (None, 1, 1, 32)          │             288 │ block1a_se_reduce[0][0]    │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 13,010,188 (49.63 MB)

 Trainable params: 8,960,617 (34.18 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# =========================
# 📌 CELL 3: Train + Save + Evaluate
# =========================

import os

MODEL_PATH = os.path.join("..", "models", "cbam_vit_efficientnet_hybrid.h5")


# -------------------------
# Load if Exists
# -------------------------
if os.path.exists(MODEL_PATH):

    print("✅ Loading saved model...")

    model = tf.keras.models.load_model(
        MODEL_PATH,
        compile=False
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

else:

    print("🚀 Training model...")

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15
    )

    model.save(MODEL_PATH)

    print("✅ Model Saved!")


# -------------------------
# Evaluation
# -------------------------
loss,acc = model.evaluate(test_ds)

print(f"\n🎯 Test Accuracy: {acc*100:.2f}%")



# -------------------------
# Learning Curves
# -------------------------
if 'history' in locals():

    plt.figure()
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title("Accuracy")
    plt.legend(["Train","Val"])
    plt.show()


    plt.figure()
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title("Loss")
    plt.legend(["Train","Val"])
    plt.show()


In [ ]:
# =========================
# 📌 CELL 4: Confusion Matrix
# =========================

import os
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Get true + predicted labels
y_true = np.concatenate([y.numpy() for x,y in test_ds])
y_true = np.argmax(y_true, axis=1)

y_pred_prob = model.predict(test_ds)
y_pred = np.argmax(y_pred_prob, axis=1)

class_names = test_ds.class_names

results_dir = os.path.join("..", "results")
os.makedirs(results_dir, exist_ok=True)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(22,18))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True
)

plt.xlabel("Predicted Label",fontsize=14)
plt.ylabel("True Label",fontsize=14)
plt.title("Confusion Matrix (CBAM + EfficientNet + ViT)",fontsize=16)

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()

plt.savefig(os.path.join(results_dir, "confusion_matrix_highres.png"), dpi=300)
plt.show()

print("✅ Saved: confusion_matrix_highres.png")


In [ ]:
# =========================
# 📌 CELL 5: ROC Curve (Macro Average)
# =========================

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize


# Binarize labels
y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

# ROC computation
fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_pred_prob.ravel())
roc_auc = auc(fpr, tpr)


plt.figure(figsize=(7,6))

plt.plot(
    fpr, tpr,
    linewidth=2,
    label=f"Macro ROC (AUC = {roc_auc:.4f})"
)

plt.plot([0,1],[0,1],'--')

plt.xlim([0,0.2])
plt.ylim([0.8,1.01])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve (All Classes)")

plt.legend()

plt.grid()

plt.tight_layout()

plt.savefig(os.path.join(results_dir, "roc_curve_highres.png"), dpi=300)
plt.show()

print("✅ Saved: roc_curve_highres.png")


In [ ]:
# =========================
# 📌 CELL 6: Learning Curves
# =========================

plt.figure(figsize=(7,5))

plt.plot(history.history['accuracy'],linewidth=2)
plt.plot(history.history['val_accuracy'],linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.legend(["Train","Validation"])

plt.grid()

plt.tight_layout()

plt.savefig(os.path.join(results_dir, "accuracy_curve_highres.png"), dpi=300)
plt.show()


plt.figure(figsize=(7,5))

plt.plot(history.history['loss'],linewidth=2)
plt.plot(history.history['val_loss'],linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training vs Validation Loss")

plt.legend(["Train","Validation"])

plt.grid()

plt.tight_layout()

plt.savefig(os.path.join(results_dir, "loss_curve_highres.png"), dpi=300)
plt.show()

print("✅ Saved accuracy & loss curves")


In [ ]:
# =========================
# 📌 CELL 7: Classification Report
# =========================

from sklearn.metrics import classification_report
import pandas as pd


report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True
)

df = pd.DataFrame(report).transpose()

df.to_csv(os.path.join(results_dir, "classification_report.csv"))

df


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ==========================
# 📌 Get True & Predicted Labels
# ==========================
y_true = np.concatenate([y.numpy() for x, y in test_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

y_pred_probs = model.predict(test_ds)   # change name if different
y_pred = np.argmax(y_pred_probs, axis=1)

class_names = test_ds.class_names
num_classes = len(class_names)

# ==========================
# 📌 Compute Confusion Matrix
# ==========================
cm = confusion_matrix(y_true, y_pred)

# ==========================
# 📌 Create Results Folder
# ==========================
results_dir = os.path.join("..", "results")
os.makedirs(results_dir, exist_ok=True)

# ==========================
# 📌 Split into Two Parts
# ==========================
mid = num_classes // 2   # 19 for 38 classes

cm_part1 = cm[:mid, :mid]
cm_part2 = cm[mid:, mid:]

labels_part1 = class_names[:mid]
labels_part2 = class_names[mid:]

# ==========================
# 📌 Plot First Half
# ==========================
plt.figure(figsize=(14,12))

sns.heatmap(
    cm_part1,
    cmap="Blues",
    xticklabels=labels_part1,
    yticklabels=labels_part1,
    fmt="d"
)

plt.title("Confusion Matrix (Classes 1–19)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, "confusion_matrix_part1.png"), dpi=300)
plt.show()
plt.close()


# ==========================
# 📌 Plot Second Half
# ==========================
plt.figure(figsize=(14,12))

sns.heatmap(
    cm_part2,
    cmap="Blues",
    xticklabels=labels_part2,
    yticklabels=labels_part2,
    fmt="d"
)

plt.title("Confusion Matrix (Classes 20–38)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, "confusion_matrix_part2.png"), dpi=300)
plt.show()
plt.close()

# ==========================
# 📌 Done
# ==========================
print("✅ Two confusion matrices saved in:")
print(results_dir)


In [2]:
pip install streamlit tensorflow pillow

  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Uninstalling protobuf-6.33.5:
      Successfully uninstalled protobuf-6.33.5
Note: you may need to restart the kernel to use updated packages.
